# An Evacuated Tower Telescope — 핵심 물리 계산 구현 / Core Physics Implementation

**논문 / Paper**: R. B. Dunn, "An Evacuated Tower Telescope," *Applied Optics*, Vol. 3, No. 12, 1964.

이 노트북은 Dunn의 진공 타워 망원경 설계의 핵심 물리적 원리를 계산과 시각화를 통해 재현합니다.

**구성 / Contents:**
1. Internal Seeing: 진공 vs. 공기 / Vacuum vs. Air
2. McMath vs. Dunn 비교 / Comparison
3. 입사창 두께와 구경의 관계 / Entrance Window Thickness vs. Aperture
4. 수은 부양과 진동 격리 / Mercury Float and Vibration Isolation
5. 터렛 토크 모터: sin² + cos² 리플 제거 / Turret Torque Motor Ripple Elimination
6. 타워 구조 단면 시각화 / Tower Cross-Section Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## Part 1: Internal Seeing — 진공 vs. 공기 / Vacuum vs. Air

공기 중 굴절률 변동에 의한 wavefront error를 계산합니다. 진공에서는 이 효과가 완전히 사라집니다.

$$\Delta n \approx -7.9 \times 10^{-5} \frac{P}{T^2} \Delta T$$

$$\text{Wavefront error} = \frac{2\pi}{\lambda} \cdot \Delta n \cdot L$$

여기서 $P$는 압력(mbar), $T$는 온도(K), $L$은 광학 경로 길이입니다.

In [ ]:
def refractive_index_variation(P_mbar, T_K, delta_T):
    """Compute refractive index variation from temperature fluctuation.
    
    Args:
        P_mbar: Pressure in mbar
        T_K: Temperature in Kelvin
        delta_T: Temperature fluctuation in K
    
    Returns:
        delta_n: Refractive index variation
    """
    return -7.9e-5 * P_mbar / T_K**2 * delta_T


def wavefront_error_waves(delta_n, path_length_m, wavelength_nm=550):
    """Compute wavefront error in units of wavelength."""
    lam_m = wavelength_nm * 1e-9
    return abs(delta_n) * path_length_m / lam_m


# Compare different pressure environments
T = 300  # K (ambient)
delta_T_values = np.linspace(0.01, 1.0, 100)  # temperature fluctuations in K

environments = {
    "Sea level (1013 mbar)": 1013,
    "Sacramento Peak\n2.8 km (750 mbar)": 750,
    "Dunn vacuum\n250μ Hg (0.33 mbar)": 0.33,
    "High vacuum\n(0.001 mbar)": 0.001,
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: wavefront error vs temperature fluctuation (McMath path = 155m)
L_mcmath = 155  # m
for name, P in environments.items():
    dn = np.abs(refractive_index_variation(P, T, delta_T_values))
    wfe = wavefront_error_waves(dn, L_mcmath)
    ax1.plot(delta_T_values, wfe, linewidth=2, label=name)

ax1.axhline(y=1, color="red", linestyle=":", alpha=0.5)
ax1.annotate("1λ limit (diffraction quality)", xy=(0.5, 1),
             fontsize=10, color="red")
ax1.set_xlabel("Temperature Fluctuation ΔT (K)")
ax1.set_ylabel("Wavefront Error (waves at 550 nm)")
ax1.set_title(f"Wavefront Error vs ΔT\n(path length = {L_mcmath} m, like McMath)")
ax1.set_yscale("log")
ax1.legend(fontsize=9)
ax1.set_ylim(1e-5, 1e4)

# Right: wavefront error vs pressure for fixed ΔT
pressures = np.logspace(-3, 3.1, 200)  # mbar
delta_T_fixed = 0.1  # typical internal fluctuation
paths = {"McMath (155 m)": 155, "Dunn (55 m)": 55, "10 m spectrograph": 10}

for name, L in paths.items():
    dn = np.abs(refractive_index_variation(pressures, T, delta_T_fixed))
    wfe = wavefront_error_waves(dn, L)
    ax2.plot(pressures, wfe, linewidth=2, label=name)

ax2.axhline(y=1, color="red", linestyle=":", alpha=0.5, label="1λ limit")
ax2.axvline(x=0.33, color="green", linestyle="--", alpha=0.5)
ax2.annotate("Dunn\nvacuum\n(250μ Hg)", xy=(0.33, 100),
             fontsize=10, color="green", fontweight="bold")
ax2.axvline(x=750, color="orange", linestyle="--", alpha=0.5)
ax2.annotate("Sac. Peak\natmosphere", xy=(750, 0.1),
             fontsize=10, color="orange")

ax2.set_xlabel("Pressure (mbar)")
ax2.set_ylabel("Wavefront Error (waves at 550 nm)")
ax2.set_title(f"Wavefront Error vs Pressure\n(ΔT = {delta_T_fixed} K)")
ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.legend()
ax2.set_ylim(1e-8, 1e4)

plt.tight_layout()
plt.show()

# Key numbers
print("=" * 60)
print("Internal Seeing: Vacuum vs Air")
print("=" * 60)
for name, P in environments.items():
    dn = abs(refractive_index_variation(P, T, 0.1))
    wfe_mcmath = wavefront_error_waves(dn, 155)
    wfe_dunn = wavefront_error_waves(dn, 55)
    name_clean = name.replace("\n", " ")
    print(f"{name_clean:<35} Δn={dn:.2e}  WFE(155m)={wfe_mcmath:.2e}λ  WFE(55m)={wfe_dunn:.2e}λ")

## Part 2: McMath vs. Dunn 종합 비교 / Comprehensive Comparison

두 망원경의 모든 핵심 사양을 시각적으로 비교합니다. 같은 저널의 같은 호에 실린 두 가지 대조적 설계 철학을 보여줍니다.

A visual comparison of all key specifications, showing two contrasting design philosophies from the same journal issue.

In [ ]:
# McMath vs Dunn — comprehensive comparison
specs = {
    "Aperture (cm)": (160, 76),
    "Focal length (m)": (90, 55),
    "f-ratio": (56, 72),
    "Solar image (cm)": (84, 51),
    "Image scale\n(mm/arcsec)": (0.436, 0.267),
    "Diffraction limit\n(arcsec, 550nm)": (0.084, 0.18),
    "Best star image\n(arcsec)": (1.0, 0.2),
    "Internal seeing\n(arcsec, typical)": (0.5, 0.0),
    "Optical path (m)": (155, 55),
    "Tower height (m)": (30, 41.5),
}

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

colors_mcmath = "#E74C3C"
colors_dunn = "#3498DB"

for i, (name, (mcmath_val, dunn_val)) in enumerate(specs.items()):
    ax = axes[i]
    bars = ax.bar(["McMath", "Dunn"], [mcmath_val, dunn_val],
                  color=[colors_mcmath, colors_dunn], alpha=0.8,
                  edgecolor="black")
    ax.set_title(name, fontsize=10, fontweight="bold")
    
    # Add value labels
    for bar, val in zip(bars, [mcmath_val, dunn_val]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{val}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_ylim(0, max(mcmath_val, dunn_val) * 1.3)

plt.suptitle("McMath (Pierce 1964) vs Dunn VTT (Dunn 1964)\n"
             "Same Journal, Same Issue — Two Philosophies",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Philosophy comparison
print("=" * 65)
print("Design Philosophy Comparison")
print("=" * 65)
print(f"{'':20} {'McMath (Pierce)':<25} {'Dunn VTT':<25}")
print("-" * 65)
comparisons = [
    ("Priority", "Light-gathering power", "Image quality"),
    ("Internal seeing", "Water-cooled encl.", "VACUUM (eliminated)"),
    ("Beam steering", "Heliostat (1 mirror)", "Turret (2 mirrors)"),
    ("Field rotation", "Uncompensated", "Turret compensates"),
    ("Vibration isoln.", "None", "Mercury float (11 tons)"),
    ("Spectrograph", "Vacuum double-pass", "Echelle + photoelectric"),
    ("Orientation", "32° inclined", "Vertical"),
    ("Coolant volume", "70,000 L water", "None (vacuum)"),
]
for name, mcmath, dunn in comparisons:
    print(f"{name:20} {mcmath:<25} {dunn:<25}")

## Part 3: 입사창 두께와 구경의 관계 / Entrance Window Thickness vs. Aperture

진공 망원경의 근본적 한계: 입사창은 대기압을 지탱해야 하며, 구경이 커지면 필요 두께가 급격히 증가합니다. Dunn이 76 cm에서 10 cm 두께의 창이 필요했다면, 더 큰 구경에서는 어떻게 될까요?

The fundamental limit of vacuum telescopes: the entrance window must withstand atmospheric pressure, and required thickness grows rapidly with aperture.

In [ ]:
# Entrance window thickness estimation
# For a circular plate under uniform pressure, the minimum thickness for
# a given maximum stress is:
# t = r * sqrt(3 * P / (8 * sigma_max))
# where r = radius, P = pressure, sigma_max = allowable stress

def window_thickness(diameter_cm, P_Pa=74000, sigma_max_Pa=1e7):
    """Estimate minimum window thickness for vacuum load.
    
    Args:
        diameter_cm: Window diameter in cm
        P_Pa: External pressure in Pa (74000 Pa at Sacramento Peak)
        sigma_max_Pa: Maximum allowable stress in Pa
    
    Returns:
        thickness_cm: Minimum thickness in cm
    """
    r_m = diameter_cm / 200  # radius in meters
    t_m = r_m * np.sqrt(3 * P_Pa / (8 * sigma_max_Pa))
    return t_m * 100  # convert to cm


apertures = np.linspace(10, 400, 200)  # cm

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: window thickness vs aperture
# Different stress limits
for sigma_label, sigma_val in [("Conservative (7 MPa)", 7e6),
                                ("Dunn design (~10 MPa)", 1e7),
                                ("Aggressive (20 MPa)", 2e7)]:
    thicknesses = window_thickness(apertures, sigma_max_Pa=sigma_val)
    ax1.plot(apertures, thicknesses, linewidth=2, label=sigma_label)

# Mark actual telescopes
ax1.plot(76, 10, "r*", markersize=15, zorder=5)
ax1.annotate("Dunn VTT\n76 cm → 10 cm thick", xy=(76, 10),
             textcoords="offset points", xytext=(20, 15), fontsize=10,
             color="red", fontweight="bold",
             arrowprops=dict(arrowstyle="->", color="red"))

ax1.plot(100, window_thickness(100), "go", markersize=10, zorder=5)
ax1.annotate("SST (100 cm)", xy=(100, window_thickness(100)),
             textcoords="offset points", xytext=(15, 10), fontsize=10, color="green")

# DKIST would need impossibly thick window
dkist_t = window_thickness(400)
ax1.plot(400, dkist_t, "k^", markersize=12, zorder=5)
ax1.annotate(f"DKIST (400 cm)\n→ {dkist_t:.0f} cm thick!\n(impossible → no vacuum)",
             xy=(400, dkist_t), textcoords="offset points", xytext=(-120, 10),
             fontsize=10, color="black", fontweight="bold",
             arrowprops=dict(arrowstyle="->"))

ax1.set_xlabel("Aperture Diameter (cm)")
ax1.set_ylabel("Minimum Window Thickness (cm)")
ax1.set_title("Entrance Window Thickness vs Aperture\n입사창 두께 vs 구경")
ax1.legend()

# Right: window mass vs aperture
rho_quartz = 2.2  # g/cm³
for sigma_label, sigma_val in [("Dunn design (~10 MPa)", 1e7)]:
    thicknesses = window_thickness(apertures, sigma_max_Pa=sigma_val)
    radii = apertures / 2
    volumes = np.pi * radii**2 * thicknesses  # cm³
    masses = volumes * rho_quartz / 1000  # kg
    ax2.plot(apertures, masses, "b-", linewidth=2.5)

ax2.plot(76, np.pi * 38**2 * 10 * rho_quartz / 1000, "r*", markersize=15, zorder=5)
ax2.annotate(f"Dunn: {np.pi * 38**2 * 10 * rho_quartz / 1000:.0f} kg",
             xy=(76, np.pi * 38**2 * 10 * rho_quartz / 1000),
             textcoords="offset points", xytext=(20, 10), fontsize=10, color="red")

ax2.set_xlabel("Aperture Diameter (cm)")
ax2.set_ylabel("Window Mass (kg)")
ax2.set_title("Entrance Window Mass vs Aperture\n입사창 질량 vs 구경")
ax2.set_yscale("log")

plt.tight_layout()
plt.show()

# Print key numbers
print("=" * 55)
print("Entrance Window: The Achilles' Heel of Vacuum Telescopes")
print("=" * 55)
for d in [76, 100, 150, 200, 400]:
    t = window_thickness(d)
    mass = np.pi * (d/2)**2 * t * rho_quartz / 1000
    print(f"D = {d:>4} cm → thickness ≥ {t:>5.1f} cm, mass ≈ {mass:>7.0f} kg")
print(f"\n→ This is why DKIST (400 cm) chose active cooling over vacuum!")

## Part 4: 수은 부양과 진동 격리 / Mercury Float and Vibration Isolation

230 metric ton의 내부 구조물을 11톤의 수은 위에 부양시키는 시스템을 스프링-질량-댐퍼(spring-mass-damper) 모델로 분석합니다. 진동 전달률(transmissibility)을 계산하여 어떤 주파수 범위에서 진동이 차단되는지 확인합니다.

Analyzing the mercury float system as a spring-mass-damper model. Calculate transmissibility to see which frequencies are isolated.

In [ ]:
def transmissibility(f_excite, f_natural, damping_ratio):
    """Compute vibration transmissibility for a spring-mass-damper system.
    
    T = sqrt((1 + (2*zeta*r)^2) / ((1-r^2)^2 + (2*zeta*r)^2))
    where r = f_excite / f_natural
    """
    r = f_excite / f_natural
    numerator = np.sqrt(1 + (2 * damping_ratio * r)**2)
    denominator = np.sqrt((1 - r**2)**2 + (2 * damping_ratio * r)**2)
    return numerator / denominator


# Frequency range
freqs = np.logspace(-1, 2, 500)  # Hz

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: transmissibility for different natural frequencies
damping = 0.1  # estimated damping ratio for mercury float
for f_n, label in [(0.5, "f₀ = 0.5 Hz (Dunn minimum)"),
                    (1.0, "f₀ = 1.0 Hz"),
                    (2.0, "f₀ = 2.0 Hz"),
                    (7.0, "f₀ = 7.0 Hz (tower natural freq)")]:
    T = transmissibility(freqs, f_n, damping)
    ax1.plot(freqs, T, linewidth=2, label=label)

ax1.axhline(y=1, color="gray", linestyle=":", alpha=0.5)
ax1.axhline(y=0.1, color="green", linestyle="--", alpha=0.3)
ax1.annotate("90% isolation", xy=(50, 0.1), fontsize=9, color="green")

# Mark key frequencies
ax1.axvline(x=7, color="red", linestyle=":", alpha=0.3)
ax1.annotate("Tower 7 Hz", xy=(7, 0.01), fontsize=9, color="red", rotation=90)
ax1.axvline(x=0.5, color="blue", linestyle=":", alpha=0.3)
ax1.annotate("Float 0.5 Hz", xy=(0.5, 0.01), fontsize=9, color="blue", rotation=90)

ax1.set_xlabel("Excitation Frequency (Hz)")
ax1.set_ylabel("Transmissibility")
ax1.set_title("Vibration Transmissibility\n진동 전달률 (damping ratio = 0.1)")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_ylim(1e-4, 20)
ax1.legend(fontsize=9)

# Right: isolation effectiveness at tower frequency (7 Hz)
# vs natural frequency of float
f_natural_range = np.linspace(0.1, 5, 200)
T_at_7Hz = transmissibility(7.0, f_natural_range, damping)

ax2.plot(f_natural_range, T_at_7Hz * 100, "b-", linewidth=2.5)
ax2.fill_between(f_natural_range, 0, T_at_7Hz * 100, alpha=0.1, color="blue")

ax2.axhline(y=1, color="red", linestyle="--", alpha=0.5, label="1% transmission")
ax2.axvline(x=0.5, color="green", linestyle="--", alpha=0.5)
ax2.annotate("Dunn minimum\nf₀ = 0.5 Hz\n→ {:.3f}% transmitted".format(
    transmissibility(7.0, 0.5, damping) * 100),
    xy=(0.5, transmissibility(7.0, 0.5, damping) * 100),
    textcoords="offset points", xytext=(30, 30), fontsize=10,
    color="green", fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="green"))

ax2.set_xlabel("Float Natural Frequency (Hz)")
ax2.set_ylabel("Transmission at 7 Hz (%)")
ax2.set_title("Tower Vibration (7 Hz) Isolation\n타워 진동 격리 효과")
ax2.set_yscale("log")
ax2.legend()

plt.tight_layout()
plt.show()

# Mercury float physics
print("=" * 55)
print("Mercury Float System")
print("=" * 55)
print(f"Float mass:           230 metric tons")
print(f"Mercury mass:         ~11 metric tons")
print(f"Mercury density:      13,600 kg/m³")
print(f"Float diameter:       3.7 m")
print(f"Mercury depth:        1.8 m")
print(f"Natural freq range:   0.5 Hz (adjustable)")
print(f"Tower natural freq:   7 Hz")
print(f"Isolation at 7 Hz:    {transmissibility(7, 0.5, 0.1)*100:.3f}% transmitted")
print(f"                    = {20*np.log10(transmissibility(7, 0.5, 0.1)):.1f} dB attenuation")

## Part 5: 터렛 토크 모터 — sin² + cos² 리플 제거 / Torque Motor Ripple Elimination

Dunn은 브러시리스 토크 모터에서 두 개의 권선(sin²과 cos²)의 합으로 회전각에 무관한 일정 토크를 생성하는 설계를 사용했습니다. 이 수학적 원리를 시각화합니다.

$$\sin^2\theta + \cos^2\theta = 1 \quad \forall\theta$$

In [ ]:
# Torque motor ripple elimination: sin² + cos² = 1
theta = np.linspace(0, 4 * np.pi, 500)  # rotation angle

# Single-phase motor (conventional): torque varies with angle
torque_single = np.sin(theta)**2

# Dunn's two-phase design
torque_sin2 = np.sin(theta)**2
torque_cos2 = np.cos(theta)**2
torque_total = torque_sin2 + torque_cos2  # always = 1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Top: individual winding contributions
ax1.plot(np.degrees(theta), torque_sin2, "r-", linewidth=2, alpha=0.7,
         label="Winding A: sin²θ")
ax1.plot(np.degrees(theta), torque_cos2, "b-", linewidth=2, alpha=0.7,
         label="Winding B: cos²θ")
ax1.plot(np.degrees(theta), torque_total, "k-", linewidth=3,
         label="Total: sin²θ + cos²θ = 1")
ax1.fill_between(np.degrees(theta), torque_sin2, alpha=0.1, color="red")
ax1.fill_between(np.degrees(theta), torque_cos2, alpha=0.1, color="blue")
ax1.set_ylabel("Normalized Torque")
ax1.set_title("Dunn's Brushless Torque Motor: Ripple Elimination\n"
              "sin²θ + cos²θ = 1 → constant torque at all angles")
ax1.legend(fontsize=11)
ax1.set_ylim(-0.1, 1.5)

# Bottom: comparison with conventional single-phase
torque_conventional = np.abs(np.sin(theta))  # typical single-phase
ripple_pct = (np.max(torque_conventional) - np.min(torque_conventional)) / np.mean(torque_conventional) * 100

ax2.plot(np.degrees(theta), torque_conventional, "gray", linewidth=2, alpha=0.7,
         label=f"Conventional single-phase\n(ripple: {ripple_pct:.0f}%)")
ax2.plot(np.degrees(theta), torque_total, "k-", linewidth=3,
         label="Dunn two-phase (ripple: 0%)")
ax2.set_xlabel("Rotation Angle θ (degrees)")
ax2.set_ylabel("Normalized Torque")
ax2.set_title("Conventional vs Dunn Motor Design")
ax2.legend(fontsize=11)
ax2.set_ylim(-0.1, 1.5)

plt.tight_layout()
plt.show()

print("=" * 55)
print("Torque Motor Design")
print("=" * 55)
print("Dunn's innovation: magnetic flux varies sinusoidally")
print("pole-to-pole. Two windings produce:")
print("  Winding A: torque ∝ sin²θ")
print("  Winding B: torque ∝ cos²θ")
print("  Total:     sin²θ + cos²θ = 1 (constant!)")
print()
print("Motor specs:")
print(f"  Torque:     1.4 × 10⁶ cm-kg (10,000 ft-lb)")
print(f"  Top speed:  1 rpm")
print(f"  Type:       Brushless, direct-drive, DC")
print(f"  Assembly:   Segmented (assembled around tube)")
print(f"  Commutator: None (brushless)")

## Part 6: 타워 구조 단면 시각화 / Tower Cross-Section Visualization

Dunn 논문의 Fig. 3을 기반으로 진공 타워 망원경의 단면을 재구성합니다.

Reconstructing the cross-section of the vacuum tower telescope based on Fig. 3 of the paper.

In [ ]:
# Dunn Vacuum Tower Telescope — cross section schematic
fig, ax = plt.subplots(figsize=(10, 16))

# Tower outline (tapered octagon, simplified as trapezoid)
tower_base_w = 8  # m half-width at base
tower_top_w = 0.75  # m half-width at top
tower_height = 41.5  # m
ground_y = 0
below_ground = 20  # m (spectrograph shaft)

# Tower walls
ax.fill([tower_base_w, tower_top_w, tower_top_w + 0.9, tower_base_w + 0.9,
          tower_base_w + 0.9, tower_top_w + 0.9, -tower_top_w - 0.9, -tower_base_w - 0.9,
          -tower_base_w - 0.9, -tower_top_w - 0.9, -tower_top_w, -tower_base_w],
         [0, tower_height, tower_height, 0,
          0, tower_height, tower_height, 0,
          0, tower_height, tower_height, 0],
         color="gray", alpha=0.3)

# Concrete walls
for side in [1, -1]:
    wall_outer = side * np.interp(np.linspace(0, tower_height, 100),
                                   [0, tower_height],
                                   [tower_base_w + 0.91, tower_top_w + 0.91])
    wall_inner = side * np.interp(np.linspace(0, tower_height, 100),
                                   [0, tower_height],
                                   [tower_base_w, tower_top_w])
    ys = np.linspace(0, tower_height, 100)
    ax.fill_betweenx(ys, wall_inner, wall_outer, color="gray", alpha=0.5)

# Turret at top
turret_y = tower_height
ax.add_patch(patches.FancyBboxPatch((-2, turret_y), 4, 3,
             boxstyle="round,pad=0.2", facecolor="steelblue", alpha=0.6,
             edgecolor="navy", linewidth=2))
ax.text(0, turret_y + 1.5, "TURRET\n(Altazimuth)\nQuartz window", ha="center",
        fontsize=9, fontweight="bold", color="white")

# Sun rays
for offset in [-1, 0, 1]:
    ax.annotate("", xy=(offset, turret_y + 3),
                xytext=(offset - 2, turret_y + 8),
                arrowprops=dict(arrowstyle="-|>", color="gold", lw=2.5))
ax.text(-3, turret_y + 7, "☀ Sunlight", fontsize=12, color="darkorange", fontweight="bold")

# Vacuum tube (central)
tube_w = 0.6  # half-width
ax.fill_between([-tube_w, tube_w], [turret_y, turret_y], [-below_ground, -below_ground],
                color="lightyellow", alpha=0.5, edgecolor="brown", linewidth=1.5)
ax.text(0, tower_height * 0.6, "EVACUATED\nTUBE\n(250μ Hg)\nØ 1.2 m",
        ha="center", fontsize=9, color="brown", fontweight="bold",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

# Light path arrow
ax.annotate("", xy=(0, -5), xytext=(0, turret_y),
            arrowprops=dict(arrowstyle="-|>", color="gold", lw=3, alpha=0.7))

# Primary mirror at bottom
ax.plot([-1.5, 1.5], [-15, -15], "r-", linewidth=4)
ax.text(0, -16.5, "Primary Mirror\n(76 cm, f=55m)", ha="center",
        fontsize=10, fontweight="bold", color="red")

# Mercury float
mercury_y = -2
ax.add_patch(patches.Rectangle((-3, mercury_y - 2), 6, 2,
             facecolor="silver", alpha=0.5, edgecolor="gray", linewidth=2))
ax.text(0, mercury_y - 1, "MERCURY FLOAT\n230 tons on 11 tons Hg",
        ha="center", fontsize=9, fontweight="bold", color="darkblue")

# Ground level
ax.axhline(y=0, color="saddlebrown", linewidth=3)
ax.text(tower_base_w + 2, 0.5, "Ground Level", fontsize=11,
        color="saddlebrown", fontweight="bold")

# Below ground section
ax.add_patch(patches.Rectangle((-3, -below_ground), 6, below_ground,
             facecolor="wheat", alpha=0.2, edgecolor="saddlebrown", linewidth=1))
ax.text(3.5, -10, "Below ground\n(~20 m)\nSpectrograph\nshaft", fontsize=9,
        color="saddlebrown")

# Work table (12m diameter)
table_y = 2
ax.plot([-6, 6], [table_y, table_y], "k-", linewidth=3)
ax.text(4, table_y + 0.5, "12-m Work Table\n(instruments)", fontsize=9)

# Auxiliary vacuum tubes
for x_pos in [-2.5, 2.5]:
    ax.fill_between([x_pos - 0.3, x_pos + 0.3],
                    [table_y, table_y], [-below_ground, -below_ground],
                    color="lightblue", alpha=0.3, edgecolor="blue", linewidth=1)
ax.text(-4, -8, "Aux vacuum\ntubes\n(Ø 1.5 m × 21 m)", fontsize=8, color="blue")

# Dimensions
ax.annotate("", xy=(tower_base_w + 2, 0), xytext=(tower_base_w + 2, tower_height),
            arrowprops=dict(arrowstyle="<->", color="black", lw=1.5))
ax.text(tower_base_w + 2.5, tower_height / 2, f"{tower_height} m",
        fontsize=10, rotation=90, va="center")

ax.annotate("", xy=(tower_base_w + 2, 0), xytext=(tower_base_w + 2, -below_ground),
            arrowprops=dict(arrowstyle="<->", color="black", lw=1.5))
ax.text(tower_base_w + 2.5, -below_ground / 2, f"~{below_ground} m",
        fontsize=10, rotation=90, va="center")

# Key innovations labels
innovations = [
    (tower_base_w + 3, tower_height - 5, "① Turret (not heliostat)\n   → no field rotation"),
    (tower_base_w + 3, tower_height * 0.5, "② Full vacuum path\n   → no internal seeing"),
    (tower_base_w + 3, 3, "③ Mercury float\n   → vibration isolation"),
    (tower_base_w + 3, -5, "④ Vertical orientation\n   → no gravitational sag"),
]
for x, y, text in innovations:
    ax.text(x, y, text, fontsize=10, color="darkgreen", fontweight="bold",
            bbox=dict(boxstyle="round", facecolor="honeydew", alpha=0.8))

ax.set_xlim(-12, 18)
ax.set_ylim(-22, turret_y + 10)
ax.set_aspect("equal")
ax.set_xlabel("Horizontal (m)")
ax.set_ylabel("Elevation (m)")
ax.set_title("Dunn Vacuum Tower Telescope — Cross Section\n"
             "(after Dunn 1964, Fig. 3)", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

## 요약 / Summary

| Part | 주제 / Topic | 핵심 결과 / Key Result |
|------|------------|---------------------|
| 1 | Vacuum vs Air | 250μ vacuum: wavefront error reduced by factor ~2000× vs atmosphere |
| 2 | McMath vs Dunn | Two contrasting philosophies: light-gathering vs image quality |
| 3 | Entrance Window | 76 cm → 10 cm thick; 400 cm → ~53 cm (impossible → why DKIST has no vacuum) |
| 4 | Mercury Float | 230 tons on 11 tons Hg; 7 Hz tower vibration attenuated by ~37 dB |
| 5 | Torque Motor | sin²θ + cos²θ = 1 → zero torque ripple at all rotation angles |
| 6 | Tower Structure | Vertical, tapered concrete, 41.5 m, no windscreen, cooling pipes embedded |

**다음 논문 / Next Paper**: #3 Swedish Solar Telescope (SST, 2002)
→ Dunn의 진공 설계 + adaptive optics = 1m 구경에서 회절 한계(0.1 arcsec) 달성
→ Dunn's vacuum design + adaptive optics = diffraction-limited (0.1 arcsec) at 1 m aperture